In [1]:
import pandas as pd
import json

In [2]:
id_and_tract = pd.read_csv("pid_and_tract.csv")
rs_keepcols = ['rs_prop_sam_id', 'rs_prop_prim_sam_id', 'rs_prop_full_address',
       'rs_prop_street_number', 'rs_prop_full_street_name',
       'rs_prop_unit_number', 'rs_prop_zip_code',
       'rs_prop_mailing_neighborhood', 'rs_prop_long', 'rs_prop_lat',
       'rs_prop_pid', 'rs_prop_lu', 'rs_prop_owner', 'rs_prop_yr_built',
       'rs_prop_yr_remod', 'rs_prop_living_area',
       'json_mhl_housing_5year', 'json_bldg_mhl_housing_5year', 'json_all_owner_props']
census_dtypedict = {}
def json_length(jsonstring):
    if len(jsonstring) == 0:
        return 0
    return len(json.loads(jsonstring))
rs_converters = {
    'json_mhl_housing_5year': json_length, 'json_bldg_mhl_housing_5year': json_length, 'json_all_owner_props': json_length
}
for colname in rs_keepcols:
    if colname[:4] == "json":
        census_dtypedict[colname] = "string" #changed to int due to converters
    elif colname in ['rs_prop_full_address', 'rs_prop_full_street_name', 'rs_prop_street_number', 'rs_prop_unit_number', 'rs_prop_mailing_neighborhood', 'rs_prop_owner']:
        census_dtypedict[colname] = "string"
    elif colname == "rs_prop_lu":
        census_dtypedict[colname] = 'category'
    elif colname in ['rs_prop_sam_id', 'rs_prop_prim_sam_id', 'rs_prop_zip_code', 'rs_prop_pid', 'rs_prop_yr_built', 'rs_prop_yr_remod', 'rs_prop_living_area', 'rs_prop_bdrms',
       'rs_prop_full_bth', 'rs_prop_half_bth']:
        census_dtypedict[colname] = 'Int64'
    else:
        census_dtypedict[colname] = 'Float64'
filenames = ["1700-to-1799.csv", "1800-to-1849.csv", "1850-to-1899.csv", "1900-to-1919.csv", "1920-to-1929.csv", "1930-to-1939.csv", "1940-to-1949.csv", "1950-to-1959.csv", "1960-to-1969.csv", "1970-to-1979.csv", "1980-to-1999.csv", "2000-to-2019.csv"]
tables = []
for f in filenames:
    tables.append(pd.read_csv(f, usecols=rs_keepcols, dtype=census_dtypedict, converters=rs_converters))
all_properties = pd.concat(tables)
print(all_properties.info(verbose=True))
# for c in all_properties.columns:
#     print(c, all_properties[c].is_unique)
for json_field in rs_converters.keys():
    print(all_properties[json_field].info(verbose=True))

C:\Users\shohi\AppData\Local\Temp\ipykernel_55224\475262810.py:32: ParserWarning: Both a converter and dtype were specified for column json_mhl_housing_5year - only the converter will be used.
  tables.append(pd.read_csv(f, usecols=rs_keepcols, dtype=census_dtypedict, converters=rs_converters))
C:\Users\shohi\AppData\Local\Temp\ipykernel_55224\475262810.py:32: ParserWarning: Both a converter and dtype were specified for column json_bldg_mhl_housing_5year - only the converter will be used.
  tables.append(pd.read_csv(f, usecols=rs_keepcols, dtype=census_dtypedict, converters=rs_converters))
C:\Users\shohi\AppData\Local\Temp\ipykernel_55224\475262810.py:32: ParserWarning: Both a converter and dtype were specified for column json_all_owner_props - only the converter will be used.
  tables.append(pd.read_csv(f, usecols=rs_keepcols, dtype=census_dtypedict, converters=rs_converters))
C:\Users\shohi\AppData\Local\Temp\ipykernel_55224\475262810.py:32: ParserWarning: Both a converter and dtype 

<class 'pandas.core.frame.DataFrame'>
Index: 367698 entries, 0 to 25626
Data columns (total 19 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   rs_prop_sam_id                367698 non-null  Int64  
 1   rs_prop_prim_sam_id           366118 non-null  Int64  
 2   rs_prop_full_address          367698 non-null  string 
 3   rs_prop_street_number         367698 non-null  string 
 4   rs_prop_full_street_name      367698 non-null  string 
 5   rs_prop_unit_number           252262 non-null  string 
 6   rs_prop_zip_code              367696 non-null  Int64  
 7   rs_prop_mailing_neighborhood  367676 non-null  string 
 8   rs_prop_long                  367698 non-null  Float64
 9   rs_prop_lat                   367698 non-null  Float64
 10  rs_prop_pid                   367698 non-null  Int64  
 11  rs_prop_lu                    367698 non-null  object 
 12  rs_prop_owner                 367698 non-null  str

In [3]:
import seaborn as sns
json_field = 'json_mhl_civic_5year'
# for json_field in rs_converters.keys():
# sns.histplot(all_properties, x=json_field)
all_properties["all_mhl"] = all_properties["json_mhl_housing_5year"]+all_properties["json_bldg_mhl_housing_5year"]

In [4]:
(all_properties["all_mhl"] == 0).mean() ## 77.5% of all properties have no Mayors hotline requests that are relevant to property owners
# sns.boxplot(all_properties, x="all_mhl")

np.float64(0.6304956785187844)

In [5]:
all_properties["rs_prop_lu"].head()
apt_buildings = all_properties[((all_properties["rs_prop_lu"] == "R2") | (all_properties["rs_prop_lu"] == "R3")) & (all_properties["rs_prop_sam_id"] == all_properties["rs_prop_prim_sam_id"])]

# sns.boxplot(apt_buildings, x="all_mhl")
print(apt_buildings.shape)
print((apt_buildings["all_mhl"] == 0).mean())


(31046, 20)
0.7856084519744895


In [6]:
def all_mhl(df):
    return df["json_mhl_housing_5year"]+["json_bldg_mhl_housing_5year"]

all_properties["all_mhl_count"] = all_properties["json_mhl_housing_5year"]+all_properties["json_bldg_mhl_housing_5year"]

In [7]:

pid_to_tract = pd.read_csv("pid_and_tract.csv", index_col="rs_prop_sam_id")
# pid_to_tract.loc[62318]["tract"]
all_properties.set_index(keys="rs_prop_sam_id", inplace=True)
all_properties["tract"] = all_properties.index.map(lambda x: pid_to_tract.loc[x]["tract"])
all_properties["tract"].info() # got a tract for every property

<class 'pandas.core.series.Series'>
Index: 367698 entries, 108715 to 83573
Series name: tract
Non-Null Count   Dtype
--------------   -----
367698 non-null  Int64
dtypes: Int64(1)
memory usage: 6.3 MB


In [8]:
census_keepcols = ["TRACTCE20", "Housing_Vehicles_share_no_vehicles", "Housing_LivingArrangements_household_pop", "Housing_LivingArrangements_group_quarters", "Housing_LivingArrangements_avg_hh_size", "Housing_Tenure_owner", "Housing_Units_total", "Housing_Units_vacancy_rate", "Housing_Units_units_per_acre", "Labor_Commute_share_worked_from_home", "Labor_LaborForce_female_participation_rate", "Labor_LaborForce_male_participation_rate", "Population_Race_other", "Population_Race_american_indian_alaska_native", "Population_Race_two_plus", "Population_Race_share_non_white", "Population_Race_share_black_african_american", "Population_Race_share_asian_pacific_islander", "Population_Race_share_hispanic_latino", "Population_Sex_female_share", "Population_Density_pop_per_sq_mi", "Population_Nativity_share_foreign_born", "Population_Age_0_to_9", "Population_Age_10_to_19", "Population_Age_20_to_34", "Population_Age_35_to_54", "Population_Age_55_to_64", "Population_Age_65_plus", "Population_Sex_total_pop", "Population_Age_share_0_to_9", "Population_Age_share_20_to_34", "Population_Age_share_55_plus", "Population_ChildrenByAge_under5", "Population_ChildrenByAge_5_to_17", "Income_Poverty_tot_pov_stat_det", "Income_Poverty_pov_rate", "Education_Attainment_share_LT_highschool", "Education_Attainment_share_bachelors_more"]

census_df = pd.read_csv("tract2020.csv", usecols=census_keepcols)
census_df.set_index(keys="TRACTCE20", inplace=True)
census_df.info

<bound method DataFrame.info of            Housing_Vehicles_share_no_vehicles  \
TRACTCE20                                       
101                                  0.196519   
102                                  0.147790   
201                                  0.109897   
202                                  0.092854   
301                                  0.123525   
...                                       ...   
981600                               0.000000   
981700                               0.000000   
981800                               0.300000   
981900                                    NaN   
981502                                    NaN   

           Housing_LivingArrangements_household_pop  \
TRACTCE20                                             
101                                      1866.00000   
102                                      4262.70295   
201                                      3914.00000   
202                                      4107.00000   


In [9]:
rs_and_census = all_properties.join(census_df, on="tract")

In [10]:
rs_and_census.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 367698 entries, 108715 to 83573
Data columns (total 58 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   rs_prop_prim_sam_id                            366118 non-null  Int64  
 1   rs_prop_full_address                           367698 non-null  string 
 2   rs_prop_street_number                          367698 non-null  string 
 3   rs_prop_full_street_name                       367698 non-null  string 
 4   rs_prop_unit_number                            252262 non-null  string 
 5   rs_prop_zip_code                               367696 non-null  Int64  
 6   rs_prop_mailing_neighborhood                   367676 non-null  string 
 7   rs_prop_long                                   367698 non-null  Float64
 8   rs_prop_lat                                    367698 non-null  Float64
 9   rs_prop_pid                           

In [11]:
isd = pd.read_csv("build_prop_viol.csv")
isd.columns
isd.info(verbose=True)
isd.status_dttm = pd.to_datetime(isd.status_dttm)
min_time = pd.to_datetime("2016-06-04 00:00:00.000000")
max_time = pd.to_datetime("2021-06-03 23:59:59.999999")
isd = isd[(isd["status_dttm"] > min_time) & (isd["status_dttm"] < max_time)]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17185 entries, 0 to 17184
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   _id               17185 non-null  int64  
 1   case_no           17185 non-null  object 
 2   ap_case_defn_key  17185 non-null  int64  
 3   status_dttm       17184 non-null  object 
 4   status            17185 non-null  object 
 5   code              17185 non-null  object 
 6   value             0 non-null      float64
 7   description       16938 non-null  object 
 8   violation_stno    17185 non-null  object 
 9   violation_sthigh  4208 non-null   object 
 10  violation_street  17185 non-null  object 
 11  violation_suffix  17040 non-null  object 
 12  violation_city    17185 non-null  object 
 13  violation_state   17185 non-null  object 
 14  violation_zip     17185 non-null  object 
 15  ward              17185 non-null  object 
 16  contact_addr1     17179 non-null  object

In [12]:
def get_code_violations(pid, prim_sam_id, metric= "all"):
    matching = isd[(isd["sam_id"] == pid) | (isd["sam_id"] == prim_sam_id)]
    if metric == "all":
        return list(matching[["_id", "case_no", "status_dttm", "status", "code", "description", "violation_stno", "violation_sthigh", "violation_street", "violation_suffix", "sam_id"]])
    elif metric == "count":
        return int(matching.shape[0])
    elif metric == "codes":
        return list(matching["code"])
    # elif metric == "sanitary":
    #     return matching[str(matching["code"])[:3] == "105"]


# print(all_properties[all_properties["rs_prop_sam_id"] == 460452])
# print(all_properties[(all_properties["rs_prop_street_number"] == "24") & (all_properties["rs_prop_full_street_name"] == "Greenville St")])
# print(all_properties.info(verbose=True))
print(isd.info(verbose=True))

<class 'pandas.core.frame.DataFrame'>
Index: 4215 entries, 3721 to 7935
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   _id               4215 non-null   int64         
 1   case_no           4215 non-null   object        
 2   ap_case_defn_key  4215 non-null   int64         
 3   status_dttm       4215 non-null   datetime64[ns]
 4   status            4215 non-null   object        
 5   code              4215 non-null   object        
 6   value             0 non-null      float64       
 7   description       4166 non-null   object        
 8   violation_stno    4215 non-null   object        
 9   violation_sthigh  1025 non-null   object        
 10  violation_street  4215 non-null   object        
 11  violation_suffix  4174 non-null   object        
 12  violation_city    4215 non-null   object        
 13  violation_state   4215 non-null   object        
 14  violation_zip     4215 non

In [13]:
# rs_and_census.to_csv("rs_with_census.csv")

In [14]:
rs_and_census.memory_usage(deep=True).sort_values(ascending=False)
wo_prop_list = rs_and_census.drop("json_all_owner_props", axis=1)
# wo_prop_list.to_csv("properties_with_census.csv")

In [15]:
# wo_prop_list["isd_violations_all"] = wo_prop_list.index.map(lambda x: get_code_violations(x, metric="all"))
wo_prop_list["isd_violations_count"] = wo_prop_list.index.map(lambda x: get_code_violations(x, wo_prop_list.loc[x]["rs_prop_prim_sam_id"], metric="count"))
wo_prop_list["isd_violation_codes"] = wo_prop_list.index.map(lambda x: get_code_violations(x, wo_prop_list.loc[x]["rs_prop_prim_sam_id"], metric="codes"))
wo_prop_list.to_csv("properties_census_isd.csv")

In [16]:
wo_prop_list.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 367698 entries, 108715 to 83573
Data columns (total 59 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   rs_prop_prim_sam_id                            366118 non-null  Int64  
 1   rs_prop_full_address                           367698 non-null  string 
 2   rs_prop_street_number                          367698 non-null  string 
 3   rs_prop_full_street_name                       367698 non-null  string 
 4   rs_prop_unit_number                            252262 non-null  string 
 5   rs_prop_zip_code                               367696 non-null  Int64  
 6   rs_prop_mailing_neighborhood                   367676 non-null  string 
 7   rs_prop_long                                   367698 non-null  Float64
 8   rs_prop_lat                                    367698 non-null  Float64
 9   rs_prop_pid                           

In [17]:
def is_building_row(df, index, Series):
    if pd.isna(Series["rs_prop_prim_sam_id"]):
        return False
    if index == Series["rs_prop_prim_sam_id"] and df[(df["rs_prop_prim_sam_id"] == index)].shape[0] != 1:
        return True
    return False

is_building_row(wo_prop_list, 31417, wo_prop_list.loc[31417])
ibr = []
for i, s in wo_prop_list.iterrows():
    ibr.append(is_building_row(wo_prop_list, i, s))
wo_prop_list["is_building_row"] = ibr
wo_prop_list = wo_prop_list[wo_prop_list["is_building_row"] == False]
wo_prop_list.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 315293 entries, 407453 to 106156
Data columns (total 60 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   rs_prop_prim_sam_id                            313713 non-null  Int64  
 1   rs_prop_full_address                           315293 non-null  string 
 2   rs_prop_street_number                          315293 non-null  string 
 3   rs_prop_full_street_name                       315293 non-null  string 
 4   rs_prop_unit_number                            252249 non-null  string 
 5   rs_prop_zip_code                               315291 non-null  Int64  
 6   rs_prop_mailing_neighborhood                   315275 non-null  string 
 7   rs_prop_long                                   315293 non-null  Float64
 8   rs_prop_lat                                    315293 non-null  Float64
 9   rs_prop_pid                          

In [18]:
wo_prop_list.all_mhl.value_counts()

all_mhl
0     192003
1      55793
2      24274
3      13679
4       7595
5       4575
6       3251
7       3002
8       2926
9       2228
10      1598
18      1315
14       608
11       488
15       431
12       378
13       321
22       256
26       131
16       101
19        97
17        39
21        31
32        26
20        25
35        16
71        15
28        11
27         9
23         9
25         9
24         8
29         8
72         5
33         5
30         4
37         3
38         2
73         2
34         2
53         2
41         1
54         1
36         1
86         1
45         1
56         1
64         1
97         1
74         1
31         1
47         1
52         1
Name: count, dtype: int64

In [19]:
wo_prop_list.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 315293 entries, 407453 to 106156
Data columns (total 60 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   rs_prop_prim_sam_id                            313713 non-null  Int64  
 1   rs_prop_full_address                           315293 non-null  string 
 2   rs_prop_street_number                          315293 non-null  string 
 3   rs_prop_full_street_name                       315293 non-null  string 
 4   rs_prop_unit_number                            252249 non-null  string 
 5   rs_prop_zip_code                               315291 non-null  Int64  
 6   rs_prop_mailing_neighborhood                   315275 non-null  string 
 7   rs_prop_long                                   315293 non-null  Float64
 8   rs_prop_lat                                    315293 non-null  Float64
 9   rs_prop_pid                          

In [ ]:
wo_prop_list["violation_count"] = wo_prop_list["all_mhl"] + wo_prop_list["isd_violations_count"]
wo_prop_list = pd.get_dummies(wo_prop_list, prefix="LU", columns=["rs_prop_lu"], drop_first=True, dtype=int)
numeric = wo_prop_list.drop(columns=["is_building_row", "rs_prop_prim_sam_id", "rs_prop_full_address", "rs_prop_street_number", "rs_prop_full_street_name", "rs_prop_unit_number", "rs_prop_zip_code", "rs_prop_mailing_neighborhood", "rs_prop_long", "rs_prop_lat", "rs_prop_pid", "rs_prop_owner", "all_mhl", "all_mhl_count", "tract", "isd_violation_codes", "is_building_row", "isd_violations_count"])


In [27]:
numeric.to_csv("numeric_prop_census_isd.csv")

In [24]:
numeric.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
Index: 315293 entries, 407453 to 106156
Data columns (total 51 columns):
 #   Column                                         Non-Null Count   Dtype  
---  ------                                         --------------   -----  
 0   rs_prop_yr_built                               315293 non-null  Int64  
 1   rs_prop_yr_remod                               272854 non-null  Int64  
 2   rs_prop_living_area                            315293 non-null  Int64  
 3   json_mhl_housing_5year                         315293 non-null  int64  
 4   json_bldg_mhl_housing_5year                    315293 non-null  int64  
 5   Housing_Vehicles_share_no_vehicles             315260 non-null  float64
 6   Housing_LivingArrangements_household_pop       315293 non-null  float64
 7   Housing_LivingArrangements_group_quarters      315293 non-null  float64
 8   Housing_LivingArrangements_avg_hh_size         315089 non-null  float64
 9   Housing_Tenure_owner                 

In [26]:
for i, S in numeric.iterrows():
    if pd.isna(S["rs_prop_yr_remod"]):
        S["rs_prop_yr_remod"] = S["rs_prop_yr_built"]
    
